In [73]:
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
import requests

Create Tool


In [ ]:
# create tool
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_currency_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor 
    between a given base and target currency
    """
    url = f"https://v6.exchangerate-api.com/v6/API_KEY/pair/{base_currency}/{target_currency}"

    response = requests.get(url)
    
    return response.json()

@tool
def convert(base_currency_value: int, conversion_factor: Annotated[float, InjectedToolArg]) -> float:
    """given a currency conversion rate this function calculate the target 
    currency value from a given base currency value
    """
    return base_currency_value * conversion_factor

In [75]:
res = get_currency_factor.invoke({"base_currency": "USD", "target_currency": "INR"})
print(res["conversion_rate"])

95.228


In [76]:
res = convert.invoke({"base_currency_value": 100, "conversion_factor": 95.228})
print(res)

9522.8


Tool Binding


In [77]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

llm_with_tools = llm.bind_tools([get_currency_factor, convert])

Tool Calling


In [85]:
messages = [HumanMessage("what is the conversion factor between INR and USD, and based on that can you convert 10 inr in usd")]

In [86]:
ai_message = llm_with_tools.invoke(messages)

In [87]:
messages.append(ai_message)

In [88]:
ai_message.tool_calls

[{'name': 'get_currency_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '3319bcfa-13ea-4467-ac3a-b15a11721228',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '4ab3af9b-54f6-4e34-9dd9-c9b01981d8af',
  'type': 'tool_call'}]

In [89]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the vlaue of conversion rate
    if tool_call["name"] == "get_currency_factor":
        tool_message1 = get_currency_factor.invoke(tool_call)
        #fetch conversion rate
        conversion_rate = json.loads(tool_message1.content)["conversion_rate"]
        messages.append(tool_message1)
    # execute the 2nd tool using the conversion rate from tool
    if tool_call["name"] == "convert":
        # fetch the current arg
        tool_call["args"]["conversion_factor"] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)


In [90]:
messages

[HumanMessage(content='what is the conversion factor between INR and USD, and based on that can you convert 10 inr in usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019df7f3-cd5f-7f80-a297-bd8da240ca2e-0', tool_calls=[{'name': 'get_currency_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '3319bcfa-13ea-4467-ac3a-b15a11721228', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_factor': 0.0105}, 'id': '4ab3af9b-54f6-4e34-9dd9-c9b01981d8af', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 154, 'output_tokens': 44, 'total_tokens': 198, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"result": "succes

In [91]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.0105. 10 INR is equal to 0.105 USD.'